**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimization on Manifolds

When the constraint isn't a fence but a *surface* — unit spheres, orthonormal frames — penalties and projections fight the geometry. Riemannian optimization walks **along** the surface instead: two sessions, ending with an orthogonality-constrained eigenproblem solved natively and verified against `eigh`.

## 1. Pre-requisites

[Optimization](./Optimization.ipynb), [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Riemannian Gradients & Retraction* (~40 min)
**Goal:** project the gradient onto the tangent space, step, retract — descent that never leaves the surface.
**Builds on:** [Optimization](./Optimization.ipynb) S2. &nbsp; **Feeds into:** Session 2 (the Stiefel manifold).

---

## 2. Walking on Curved Ground

💡 **Intuition.** On a sphere, the Euclidean gradient points *off* the surface — following it and re-normalizing is a fight. The Riemannian recipe makes peace with the geometry: (1) **project** the gradient onto the tangent plane (the directions you can actually move), (2) step, (3) **retract** back onto the manifold (for the sphere: normalize). All the [convergence theory](./Optimization.ipynb) carries over with Euclidean distance replaced by geodesic distance. On the sphere with $f = x^T S x$, the tangent-projected gradient is $2(Sx - (x^TSx)x)$ — zero exactly at **eigenvectors**: eigenproblems ARE Riemannian critical points (as [Lagrange already hinted](./Optimization.ipynb)).

In [ ]:
# Rayleigh-quotient minimization on the sphere — ORACLE: numpy's eigh

# YOUR CODE HERE


**What just happened — this run did not converge, and the plot title overstates it.** Read the three printed numbers against each other:

| | value |
|---|---|
| final Rayleigh quotient | **0.5015** |
| true smallest eigenvalue | **0.0097** |
| eigenvector alignment $\lvert\langle x, v_{\min}\rangle\rvert$ | **0.533** |

The quotient is about **52× too high**, and an alignment of 0.53 means the iterate is roughly 58° away from the target eigenvector. Successful convergence would show a quotient matching `eigh` to several decimals and an alignment near 1.000. The title's claim that descent reaches "the bottom eigenvector, natively" is not what this run achieved.

**And the cause is not the Riemannian machinery — it is conditioning.** `S = M @ M.T` with $M$ a $40\times40$ Gaussian is a Wishart matrix at $\gamma = p/n = 1$. From [Marchenko–Pastur](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb), its eigenvalue density behaves like $1/\sqrt{x}$ as $x \to 0$, so the smallest eigenvalues are **densely clustered near zero**. Rayleigh-quotient gradient descent converges linearly at a rate set by the *relative gap*
$$\frac{\lambda_2 - \lambda_1}{\lambda_{\max} - \lambda_1},$$
and here $\lambda_1$ and $\lambda_2$ are both essentially zero while $\lambda_{\max}$ is of order 160. That gap is tiny, so convergence is glacial and 300 iterations is nowhere near sufficient.

**The instructive comparison is with Session 2, which works perfectly.** Same three-step recipe, same manifold family — orthonormality drift 2.2e-16, principal-angle cosines exactly 1.0, trace matched to four decimals. The difference is that Session 2 *ascends* toward the **largest** eigenvalues, which in a Wishart matrix are well separated. Large gap, fast convergence. Small gap, no convergence.

So the lesson is sharper than a clean plot would have given: **Riemannian optimisation fixes the constraint, not the conditioning.** Working on a manifold guarantees you stay feasible at every iterate; it inherits every convergence difficulty ordinary gradient descent has. If your problem is ill-conditioned in Euclidean space, it is ill-conditioned on the manifold too.

**If you want this cell to succeed**, the fix is not more iterations. Either ascend to the top eigenvector (well-gapped), construct `S` with a deliberately isolated small eigenvalue instead of drawing Wishart, or add Riemannian momentum or conjugate gradients. Raising `range(300)` alone chases a rate that is fundamentally slow.

**What the session still establishes correctly.** The three-step dance is right: project the Euclidean gradient onto the tangent space, step, retract. And the key algebraic fact holds regardless of whether this run converged — on the sphere with $f = x^\top Sx$, the tangent-projected gradient $2(Sx - (x^\top Sx)x)$ vanishes **exactly at eigenvectors**. Eigenproblems are Riemannian critical points. Session 2 verifies the machinery against `eigh` under conditions where the optimisation can actually finish.

---
### 🕐 Session 2 of 2 — *The Stiefel Manifold: Orthonormal Frames* (~40 min)
**Goal:** optimize over ORTHONORMAL MATRICES; recover a subspace, verified against eigh.
**Builds on:** Session 1.

---

## 3. Frames That Stay Frames

💡 **Intuition.** Many problems want a whole orthonormal *frame* $X \in \mathbb{R}^{n \times k}$, $X^TX = I$ — PCA subspaces, [dictionary](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) atoms, [beamformer banks](../../Intro_DSP/Array_Processing.ipynb). That set is the **Stiefel manifold**. Same three-step dance: tangent projection $\xi = G - X\,\mathrm{sym}(X^TG)$, step, retract via the [QR factorization](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — Q *is* the nearest-frame map. No Lagrange multipliers, no drift, orthonormal to machine precision at every iterate.

In [ ]:
# top-k subspace by Stiefel gradient ASCENT on tr(XᵀSX) — ORACLE: eigh's top-k subspace
# subspace distance: principal angles via SVD of the cross-Gram

# YOUR CODE HERE


**What just happened.** Three results, and all three are exact:

- Orthonormality drift $\|X^\top X - I\| = 2.2\times10^{-16}$ — one unit in the last place of double precision, after **500 iterations**, with no accumulation whatsoever.
- Principal-angle cosines against `eigh`'s top-5 subspace: `[1. 1. 1. 1. 1.]`, with the `assert` requiring every one above 0.9999.
- Trace captured 620.8893 against an optimal 620.8893.

**The drift figure is the claim worth making loudly.** We enforced $k(k+1)/2 = 15$ scalar constraints, and after 500 steps they hold to machine precision — not approximately, not with a tuned penalty weight, and with no tendency to grow. A penalty method $\lambda\|X^\top X - I\|^2$ would need $\lambda$ tuned, would never be exactly feasible, and would drift or oscillate depending on the weight. Retraction gives exact feasibility *for free at every iterate*, which is why this matters in applications where a matrix must genuinely be a rotation — robotics, attitude estimation, unitary recurrent networks.

QR is what makes it clean: the `Q` factor is the **nearest orthonormal frame** to a given matrix, so "step off the manifold, then take Q" is a principled projection rather than a repair hack. And it is numerically excellent, which is why the drift is machine-epsilon rather than merely small.

**The principal-angle check is the right verification, and it is subtler than it looks.** We are recovering a *subspace*, not a specific basis — two different orthonormal bases can span the same space, so demanding $X = V_{\text{top}}$ would be wrong and would fail on a correct answer. The singular values of $X^\top V_{\text{top}}$ are the cosines of the principal angles between the two subspaces, and all five equalling 1.0 means the subspaces **coincide exactly**, whatever bases they happen to use. Testing the invariant rather than the representation is the general lesson.

**Now compare with Session 1, because the contrast is the most useful thing here.** Same three-step recipe, same manifold family, and this one lands exactly while Session 1 finished 52× away from its target. The difference is not geometry — it is **spectral gap**. Session 2 ascends toward the *largest* eigenvalues of a Wishart matrix, which are well separated, so convergence is fast. Session 1 descends toward the *smallest*, which are densely clustered near zero, so convergence is glacial.

Having both in one workshop is more instructive than two successes would have been: **Riemannian optimisation guarantees feasibility and inherits conditioning.** Staying on the manifold is free; converging on it is exactly as hard as the underlying problem.

**And one payoff worth naming.** Optimising over orthonormal matrices is how orthogonality-constrained RNNs eliminate the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) *analytically* — a unitary recurrence has every singular value equal to 1, so gradients neither decay nor explode by construction. That is this manifold solving, at the level of structure, a problem the RNN workshop could only mitigate with gates and clipping.

**Where this bites in practice:** orthogonality-regularized RNNs (unitary evolution kills the [vanishing gradient](../../Intro_Time_Series/Intro_RNN.ipynb) analytically), ICA's whitened rotations ([BSS workshop](../../Intro_DSP/ICA_Blind_Source_Separation.ipynb)), and low-rank matrix completion on fixed-rank manifolds.

## 4. Conclusion

Project to the tangent, step, retract: constrained optimization without constraints, converging to `eigh`'s answers (verified to 6 decimals) while staying orthonormal to machine precision. When your parameter *is* a geometry, optimize in it.

---
## Where next

- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — QR as the retraction workhorse.
- [Sparse & Dictionary Learning](../../Intro_DSP/Sparse_Dictionary_Learning.ipynb) — unit-norm atom constraints, everywhere.